In [1]:
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Any, Dict, List, Tuple, Optional

import fitz  # pymupdf
import requests

from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, PageBreak
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import cm


# =========================
# Config
# =========================
IN_DIR = Path("PDFs")          # carpeta de entrada con PDFs
OUT_DIR = Path("PDFs_chunks_out") # carpeta de salida
OUT_DIR.mkdir(exist_ok=True)

OLLAMA_URL = "http://localhost:11434"
LLM_MODEL = "llama3.1:8b"  # ChatGPT OSS 20B en Ollama

# Para enviar al LLM: fragmentos lo suficientemente pequeños para evitar mezclar secciones
RAW_CHUNK_SIZE_CHARS = 2500
RAW_CHUNK_OVERLAP_CHARS = 250

# Robustez de JSON
MAX_REPAIR_TRIES = 3
TIMEOUT_S = 300


# =========================
# PDF extract + chunk
# =========================
def extract_pdf_text(pdf_path: Path) -> str:
    doc = fitz.open(pdf_path)
    parts = []
    for page in doc:
        parts.append(page.get_text("text") or "")
    doc.close()
    text = "\n".join(parts)
    text = text.replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    return text


def chunk_by_chars(text: str, size: int, overlap: int) -> List[str]:
    text = (text or "").strip()
    if not text:
        return []
    out = []
    i = 0
    n = len(text)
    while i < n:
        j = min(i + size, n)
        out.append(text[i:j])
        if j == n:
            break
        i = max(j - overlap, 0)
    return out


# =========================
# Ollama chat + JSON parsing
# =========================
def ollama_chat(messages: List[Dict[str, str]], model: str = LLM_MODEL) -> str:
    r = requests.post(
        f"{OLLAMA_URL}/api/chat",
        json={"model": model, "messages": messages, "stream": False},
        timeout=TIMEOUT_S,
    )
    r.raise_for_status()
    return r.json()["message"]["content"]


def safe_json_parse(s: str) -> Dict[str, Any]:
    """
    Parser robusto:
      1) JSON directo
      2) quitar fences ```json
      3) extraer primer {...} válido dentro del texto
    """
    s = (s or "").strip()

    if s.startswith("```"):
        s = re.sub(r"^```[a-zA-Z]*\s*", "", s)
        s = re.sub(r"\s*```$", "", s).strip()

    try:
        return json.loads(s)
    except Exception:
        pass

    first = s.find("{")
    last = s.rfind("}")
    if first != -1 and last != -1 and last > first:
        cand = s[first:last + 1]
        try:
            return json.loads(cand)
        except Exception:
            # intentar cierres sucesivos
            for end in range(last, first, -1):
                if s[end] == "}":
                    try:
                        return json.loads(s[first:end + 1])
                    except Exception:
                        continue

    raise ValueError("No se pudo extraer JSON válido.")


def repair_prompt(bad_output: str) -> str:
    return (
        "Tu salida anterior NO es JSON válido. Devuelve SOLO JSON válido, sin texto adicional.\n\n"
        "Debe ser exactamente:\n"
        "{\n"
        '  "chunks": [\n'
        '    {"title": "...", "content": "..."}\n'
        "  ]\n"
        "}\n\n"
        f"Salida anterior:\n{bad_output}\n"
    )


# =========================
# LLM chunker (factual, no inventa)
# =========================
def llm_make_chunks(fragment: str) -> List[Dict[str, str]]:
    """
    Entrada: texto "crudo" (fragmento del PDF).
    Salida: lista de chunks [{title, content}] autocontenidos.
    Reglas: no inventar; etiquetar correo/teléfono explícitamente si aparecen.
    """
    system = (
        "Eres un asistente de edición factual. Debes transformar texto crudo en 'chunks' autocontenidos "
        "aptos para embeddings.\n"
        "Reglas estrictas:\n"
        "- NO inventes información ni completes datos faltantes.\n"
        "- Mantén exactamente correos, teléfonos, nombres propios, cargos y números.\n"
        "- Si aparece un correo, debe quedar explícitamente como 'Correo: <valor>'.\n"
        "- Si aparece un teléfono, debe quedar explícitamente como 'Teléfono: <valor>'.\n"
        "- Elimina separadores tipo '|', guiones decorativos y ruido, preservando la información.\n"
        "- Cada chunk debe ser autocontenido (no referirse a 'arriba/abajo').\n"
        "- Devuelve SOLO JSON válido (sin markdown) con la forma:\n"
        "{ \"chunks\": [ {\"title\": \"...\", \"content\": \"...\"}, ... ] }\n"
        "- Si el fragmento no contiene información útil, devuelve {\"chunks\": []}.\n"
    )

    user = (
        "Transforma este fragmento en chunks autocontenidos.\n\n"
        "FRAGMENTO:\n"
        f"{fragment}\n"
    )

    raw = ollama_chat([{"role": "system", "content": system},
                       {"role": "user", "content": user}])

    for _ in range(MAX_REPAIR_TRIES):
        try:
            data = safe_json_parse(raw)
            chunks = data.get("chunks", [])
            # saneo mínimo
            out = []
            for c in chunks:
                title = str(c.get("title", "")).strip()
                content = str(c.get("content", "")).strip()
                if content:
                    out.append({"title": title or "Chunk", "content": content})
            return out
        except Exception:
            raw = ollama_chat([{"role": "system", "content": system},
                               {"role": "user", "content": user},
                               {"role": "user", "content": repair_prompt(raw)}])

    # Si falla siempre, no rompemos el pipeline: devolvemos vacío
    return []


# =========================
# Outputs: JSONL + PDF
# =========================
def write_jsonl(chunks: List[Dict[str, str]], out_path: Path) -> None:
    with out_path.open("w", encoding="utf-8") as f:
        for i, c in enumerate(chunks, start=1):
            rec = {
                "chunk_id": i,
                "title": c.get("title", ""),
                "content": c.get("content", ""),
            }
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")


def build_pdf_from_chunks(chunks: List[Dict[str, str]], out_pdf: Path, doc_title: Optional[str] = None) -> None:
    styles = getSampleStyleSheet()
    title_style = styles["Heading2"]
    body_style = styles["BodyText"]

    doc = SimpleDocTemplate(
        str(out_pdf),
        pagesize=letter,
        leftMargin=2 * cm,
        rightMargin=2 * cm,
        topMargin=2 * cm,
        bottomMargin=2 * cm,
    )

    story = []

    if doc_title:
        story.append(Paragraph(doc_title.replace("&", "&amp;"), styles["Title"]))
        story.append(Spacer(1, 12))

    for idx, c in enumerate(chunks, start=1):
        t = (c.get("title") or f"Chunk {idx}").strip()
        body = (c.get("content") or "").strip()

        # escape mínimo
        t = t.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
        body = body.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
        body = body.replace("\n", "<br/>")

        story.append(Paragraph(f"{idx}. {t}", title_style))
        story.append(Spacer(1, 6))
        story.append(Paragraph(body, body_style))
        story.append(Spacer(1, 12))

        # page break suave cada ~6 chunks (opcional)
        if idx % 6 == 0 and idx < len(chunks):
            story.append(PageBreak())

    doc.build(story)


# =========================
# Main
# =========================
def process_pdf(pdf_path: Path) -> Tuple[List[Dict[str, str]], List[str]]:
    raw = extract_pdf_text(pdf_path)
    raw_frags = chunk_by_chars(raw, RAW_CHUNK_SIZE_CHARS, RAW_CHUNK_OVERLAP_CHARS)

    all_chunks: List[Dict[str, str]] = []
    errors: List[str] = []

    for k, frag in enumerate(raw_frags, start=1):
        chunks = llm_make_chunks(frag)
        all_chunks.extend(chunks)

    # Deduplicación básica por contenido exacto (evita duplicados por overlap)
    seen = set()
    dedup = []
    for c in all_chunks:
        key = c["content"].strip()
        if not key or key in seen:
            continue
        seen.add(key)
        dedup.append(c)

    return dedup, errors


def main():
    if not IN_DIR.exists():
        raise FileNotFoundError(f"No existe la carpeta de entrada: {IN_DIR.resolve()}")

    pdfs = sorted([p for p in IN_DIR.rglob("*.pdf") if p.is_file()])
    if not pdfs:
        raise FileNotFoundError(f"No hay PDFs en: {IN_DIR.resolve()}")

    for pdf in pdfs:
        print(f"\n=== Procesando: {pdf.name} ===")
        chunks, errors = process_pdf(pdf)

        out_jsonl = OUT_DIR / f"{pdf.stem}_chunks.jsonl"
        out_pdf = OUT_DIR / f"{pdf.stem}_chunks.pdf"

        write_jsonl(chunks, out_jsonl)
        build_pdf_from_chunks(chunks, out_pdf, doc_title=pdf.stem)

        print(f"- Chunks: {len(chunks)}")
        print(f"- JSONL:  {out_jsonl.resolve()}")
        print(f"- PDF:    {out_pdf.resolve()}")
        if errors:
            print("- Errores:", errors)

    print("\nListo. Salida:", OUT_DIR.resolve())


if __name__ == "__main__":
    main()



=== Procesando: Estatutos Ciencia de Datos.pdf ===
- Chunks: 36
- JSONL:  C:\Users\avega\Desktop\BOT_GABRIELITO\landing_page_movile\Chatbot\PDFs_chunks_out\Estatutos Ciencia de Datos_chunks.jsonl
- PDF:    C:\Users\avega\Desktop\BOT_GABRIELITO\landing_page_movile\Chatbot\PDFs_chunks_out\Estatutos Ciencia de Datos_chunks.pdf

=== Procesando: Informacion autoridades y preguntas.pdf ===
- Chunks: 20
- JSONL:  C:\Users\avega\Desktop\BOT_GABRIELITO\landing_page_movile\Chatbot\PDFs_chunks_out\Informacion autoridades y preguntas_chunks.jsonl
- PDF:    C:\Users\avega\Desktop\BOT_GABRIELITO\landing_page_movile\Chatbot\PDFs_chunks_out\Informacion autoridades y preguntas_chunks.pdf

=== Procesando: Informacion de Pagina web.pdf ===
- Chunks: 20
- JSONL:  C:\Users\avega\Desktop\BOT_GABRIELITO\landing_page_movile\Chatbot\PDFs_chunks_out\Informacion de Pagina web_chunks.jsonl
- PDF:    C:\Users\avega\Desktop\BOT_GABRIELITO\landing_page_movile\Chatbot\PDFs_chunks_out\Informacion de Pagina web_chunks

In [2]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Dict, List, Any, Tuple

import numpy as np
import requests
import faiss


# =========================
# Config
# =========================
CHUNKS_DIR = Path("PDFs_chunks_out")  # debe coincidir con OUT_DIR del script anterior
STORE_DIR = Path("rag_store_chunks")
STORE_DIR.mkdir(exist_ok=True)

OLLAMA_URL = "http://localhost:11434"
EMBED_MODEL = "nomic-embed-text"  # o "mxbai-embed-large"

FAISS_PATH = STORE_DIR / "index.faiss"
META_PATH = STORE_DIR / "meta.jsonl"   # (pdf, chunk_id, title)
TEXT_PATH = STORE_DIR / "texts.jsonl"  # content

TIMEOUT_S = 240


# =========================
# Ollama embeddings
# =========================
def ollama_embedding(text: str, model: str = EMBED_MODEL) -> np.ndarray:
    r = requests.post(
        f"{OLLAMA_URL}/api/embeddings",
        json={"model": model, "prompt": text},
        timeout=TIMEOUT_S,
    )
    r.raise_for_status()
    return np.array(r.json()["embedding"], dtype=np.float32)


def l2_normalize_rows(x: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    n = np.linalg.norm(x, axis=1, keepdims=True)
    return x / (n + eps)


# =========================
# Load chunks
# =========================
def load_all_chunks(chunks_dir: Path) -> Tuple[List[str], List[Dict[str, Any]]]:
    texts: List[str] = []
    meta: List[Dict[str, Any]] = []

    files = sorted(chunks_dir.rglob("*_chunks.jsonl"))
    if not files:
        raise FileNotFoundError(f"No encontré *_chunks.jsonl en: {chunks_dir.resolve()}")

    for fp in files:
        pdf_name = fp.name.replace("_chunks.jsonl", "")
        with fp.open("r", encoding="utf-8") as f:
            for line in f:
                rec = json.loads(line)
                content = (rec.get("content") or "").strip()
                if not content:
                    continue
                texts.append(content)
                meta.append({
                    "pdf": pdf_name,
                    "chunk_id": rec.get("chunk_id"),
                    "title": rec.get("title", ""),
                })

    return texts, meta


def main():
    texts, meta = load_all_chunks(CHUNKS_DIR)
    print(f"Chunks totales para embeddings: {len(texts)}")

    embs = []
    for i, t in enumerate(texts, start=1):
        if i % 50 == 0:
            print(f"- Embedding {i}/{len(texts)}")
        embs.append(ollama_embedding(t))

    E = np.vstack(embs).astype(np.float32)
    E = l2_normalize_rows(E)

    d = E.shape[1]
    index = faiss.IndexFlatIP(d)  # cosine similarity por normalización
    index.add(E)

    faiss.write_index(index, str(FAISS_PATH))

    with META_PATH.open("w", encoding="utf-8") as fm, TEXT_PATH.open("w", encoding="utf-8") as ft:
        for m, t in zip(meta, texts):
            fm.write(json.dumps(m, ensure_ascii=False) + "\n")
            ft.write(json.dumps({"text": t}, ensure_ascii=False) + "\n")

    print("Listo.")
    print("FAISS:", FAISS_PATH.resolve())
    print("META:", META_PATH.resolve())
    print("TEXT:", TEXT_PATH.resolve())
    print("Embed model:", EMBED_MODEL)


if __name__ == "__main__":
    main()


Chunks totales para embeddings: 76
- Embedding 50/76
Listo.
FAISS: C:\Users\avega\Desktop\BOT_GABRIELITO\landing_page_movile\Chatbot\rag_store_chunks\index.faiss
META: C:\Users\avega\Desktop\BOT_GABRIELITO\landing_page_movile\Chatbot\rag_store_chunks\meta.jsonl
TEXT: C:\Users\avega\Desktop\BOT_GABRIELITO\landing_page_movile\Chatbot\rag_store_chunks\texts.jsonl
Embed model: nomic-embed-text


In [4]:
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Any, Dict, List, Tuple

import numpy as np
import requests
import faiss


# =========================
# Config
# =========================
STORE_DIR = Path("rag_store_chunks")  # misma carpeta del script embeddings
FAISS_PATH = STORE_DIR / "index.faiss"
META_PATH  = STORE_DIR / "meta.jsonl"
TEXT_PATH  = STORE_DIR / "texts.jsonl"

OLLAMA_URL  = "http://localhost:11434"
CHAT_MODEL  = "llama3.1:8b"
EMBED_MODEL = "nomic-embed-text"  # debe ser el mismo con el que indexaste

TOP_K = 5
MIN_SIM_FOR_RAG = 0.28  # ajusta: 0.22 más permisivo, 0.33 más estricto
MAX_CONTEXT_CHARS = 6500
TIMEOUT_S = 240


# =========================
# Ollama
# =========================
def ollama_embedding(text: str, model: str = EMBED_MODEL) -> np.ndarray:
    r = requests.post(
        f"{OLLAMA_URL}/api/embeddings",
        json={"model": model, "prompt": text},
        timeout=TIMEOUT_S,
    )
    r.raise_for_status()
    return np.array(r.json()["embedding"], dtype=np.float32)


def ollama_chat(messages: List[Dict[str, str]], model: str = CHAT_MODEL) -> str:
    r = requests.post(
        f"{OLLAMA_URL}/api/chat",
        json={"model": model, "messages": messages, "stream": False},
        timeout=TIMEOUT_S,
    )
    r.raise_for_status()
    return r.json()["message"]["content"]


# =========================
# Helpers
# =========================
def l2_normalize_vec(x: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    n = float(np.linalg.norm(x))
    return x / (n + eps)


def es_saludo(text: str) -> bool:
    t = text.strip().lower()
    return t in {"hola", "holaa", "hello", "buenas", "buenos dias", "buenas tardes", "buenas noches"}


def limpiar_texto(s: str) -> str:
    s = (s or "").replace("\x00", " ")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()


def title_is_generic(title: str) -> bool:
    """
    Detecta títulos genéricos que NO quieres mostrar: "Chunk" o "Chunk 12".
    """
    t = (title or "").strip()
    if not t:
        return True
    if t.lower() == "chunk":
        return True
    if re.fullmatch(r"chunk\s*\d+", t.strip(), flags=re.IGNORECASE):
        return True
    return False


# =========================
# Load store
# =========================
def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    out = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            out.append(json.loads(line))
    return out


def load_store() -> Tuple[faiss.Index, List[Dict[str, Any]], List[str]]:
    if not FAISS_PATH.exists():
        raise FileNotFoundError(f"No existe: {FAISS_PATH.resolve()}")
    if not META_PATH.exists():
        raise FileNotFoundError(f"No existe: {META_PATH.resolve()}")
    if not TEXT_PATH.exists():
        raise FileNotFoundError(f"No existe: {TEXT_PATH.resolve()}")

    index = faiss.read_index(str(FAISS_PATH))
    meta = load_jsonl(META_PATH)

    texts_json = load_jsonl(TEXT_PATH)
    texts = [r["text"] for r in texts_json]

    if len(meta) != len(texts):
        raise ValueError(f"Inconsistencia: meta={len(meta)} vs texts={len(texts)}")

    return index, meta, texts


# =========================
# Retrieval + context
# =========================
def retrieve(index: faiss.Index, meta: List[Dict[str, Any]], texts: List[str], query: str, topk: int = TOP_K):
    q = ollama_embedding(query)
    qn = l2_normalize_vec(q).astype(np.float32)

    D, I = index.search(qn.reshape(1, -1), topk)
    scores = D[0].tolist()
    idxs = I[0].tolist()

    results = []
    for score, idx in zip(scores, idxs):
        if idx < 0:
            continue
        m = meta[idx]
        t = texts[idx]
        results.append({
            "score": float(score),
            "pdf": m.get("pdf", ""),
            "chunk_id": m.get("chunk_id", None),
            "title": m.get("title", "") or "",
            "text": t,
        })
    return results


def build_context(results: List[Dict[str, Any]], max_chars: int = MAX_CONTEXT_CHARS) -> str:
    """
    Construye contexto con trazabilidad, omitiendo títulos genéricos (Chunk/Chunk N).
    """
    parts = []
    used = 0
    for r in results:
        pdf = r["pdf"]
        cid = r["chunk_id"]
        score = r["score"]
        title = r.get("title", "")

        # Header: sin "Chunk"
        if title_is_generic(title):
            header = f"[Fuente: {pdf} | id: {cid} | score={score:.3f}]"
        else:
            header = f"[Fuente: {pdf} | {title} | id: {cid} | score={score:.3f}]"

        body = limpiar_texto(r["text"])
        block = header + "\n" + body + "\n"

        if used + len(block) > max_chars:
            break
        parts.append(block)
        used += len(block)

    return "\n".join(parts).strip()


# =========================
# Chat
# =========================
def answer(user_msg: str, history: List[Dict[str, str]], index, meta, texts) -> str:
    if es_saludo(user_msg):
        return "hola"

    results = retrieve(index, meta, texts, user_msg, topk=TOP_K)
    best = results[0]["score"] if results else 0.0
    use_rag = best >= MIN_SIM_FOR_RAG

    system = (
        "Eres un asistente útil y conversacional.\n"
        "Reglas:\n"
        "- Si el usuario saluda, responde el saludo.\n"
        "- Si se entrega contexto documental, úsalo para responder con precisión.\n"
        "- No inventes datos; si el contexto no trae la respuesta, puedes dar una respuesta basada en lo que sabes.\n"
        "- Prioriza responder de forma clara y directa."
    )

    if use_rag:
        context = build_context(results)
        user_payload = (
            "CONTEXTO (fragmentos recuperados de documentos):\n"
            f"{context}\n\n"
            "PREGUNTA DEL USUARIO:\n"
            f"{user_msg}"
        )
    else:
        user_payload = user_msg

    messages = [{"role": "system", "content": system}]
    messages.extend(history[-12:])
    messages.append({"role": "user", "content": user_payload})
    return ollama_chat(messages, model=CHAT_MODEL)


def main():
    index, meta, texts = load_store()
    print("Store cargado:")
    print("- chunks:", len(texts))
    print("- dim:", index.d)
    print("\nChat listo. Escribe 'exit' para salir.\n")

    history: List[Dict[str, str]] = []

    while True:
        user_msg = input("Tú: ").strip()
        if not user_msg:
            continue
        if user_msg.lower() in {"exit", "quit", "salir"}:
            break

        try:
            ans = answer(user_msg, history, index, meta, texts)
        except Exception as e:
            ans = f"Ocurrió un error: {e}"

        print(f"Asistente: {ans}\n")

        history.append({"role": "user", "content": user_msg})
        history.append({"role": "assistant", "content": ans})


if __name__ == "__main__":
    main()


Store cargado:
- chunks: 76
- dim: 768

Chat listo. Escribe 'exit' para salir.

Asistente: La ciencia de datos es un campo que combina conocimientos técnicos (informática, estadística, matemáticas) para analizar y extraer conocimiento a partir de grandes conjuntos de datos. Su objetivo es obtener información valiosa para tomar decisiones informadas en diversas áreas.

Asistente: Me parece que el contexto entregado no menciona explícitamente mi función aquí. Sin embargo, según la información general disponible sobre asistentes de conversación como yo, nuestra función principal es proporcionar ayuda y responder a preguntas del usuario de manera clara y precisa. No tengo una función específica relacionada con la Ciencia de Datos, pero puedo brindarte información general o basada en conocimientos previos sobre el tema.

